# 感知机：深度学习的老祖宗
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/分类 | 源文件：感知机.py | 核心功能：线性分类、在线学习（partial_fit）、正则化对比

## 概述

感知机（Perceptron）是 1957 年由 Frank Rosenblatt 提出的——它是**人工神经网络的前身**，也是深度学习的历史起点。一个没有隐藏层的感知机，本质上就是一个线性二分类器：y = sign(w·x + b)。

虽然单层感知机只能处理线性可分问题（Minsky 在 1969 年证明了它连 XOR 都学不了），但它的核心思想——通过误分类样本迭代更新权重——直接催生了多层感知机（MLP）和现代深度学习。

脚本演示了感知机的训练、学习率调参、正则化、在线学习（partial_fit），以及它在非线性问题上的局限性。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
# --- 导入库 ---
import numpy as np

## 数学原理

### 1. 感知机模型

**代码对应**：`Perceptron(eta0=1.0, max_iter=1000)` 训练感知机。

感知机是最简单的神经网络——单个神经元，无隐藏层。模型为：

$$f(\mathbf{x}) = \text{sign}(\mathbf{w}^T\mathbf{x} + b) = \begin{cases} +1 & \text{if } \mathbf{w}^T\mathbf{x} + b \geq 0 \\ -1 & \text{otherwise} \end{cases}$$

### 2. 感知机学习算法

**代码对应**：sklearn 内部使用 SGD 优化。

感知机的训练规则：对每个误分类样本 $(\mathbf{x}_i, y_i)$（即 $y_i(\mathbf{w}^T\mathbf{x}_i + b) \leq 0$），更新：

$$\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot y_i \mathbf{x}_i$$

$$b \leftarrow b + \eta \cdot y_i$$

其中 $\eta$ 为学习率（`eta0` 参数）。

**直觉**：如果正样本被误分为负类（$\mathbf{w}^T\mathbf{x} + b < 0$），则将 $\mathbf{w}$ 向 $\mathbf{x}$ 方向调整，使 $\mathbf{w}^T\mathbf{x}$ 增大。

### 3. 收敛性定理（Perceptron Convergence Theorem）

**Novikoff 定理**：如果训练数据**线性可分**（存在 $\mathbf{w}^*$ 使得 $y_i(\mathbf{w}^{*T}\mathbf{x}_i + b^*) > 0$ 对所有 $i$ 成立），则感知机算法在有限步内收敛。

设 $R = \max_i\|\mathbf{x}_i\|$，$\gamma = \min_i y_i(\mathbf{w}^{*T}\mathbf{x}_i + b^*)/\|\mathbf{w}^*\|$（几何间隔），则最多需要：

$$k \leq \frac{R^2}{\gamma^2}$$

次更新即可收敛。

**局限**：如果数据**不**线性可分（如 XOR 问题），感知机**永远不会收敛**——权重会持续振荡。

**代码对应**：代码中尝试用感知机处理非线性可分数据，展示了其局限性。

### 4. 与逻辑回归的区别

| 特性 | 感知机 | 逻辑回归 |
|------|--------|---------|
| 输出 | 类别标签 $\{-1, +1\}$ | 概率 $P(y=1|\mathbf{x}) \in (0,1)$ |
| 损失函数 | 误分类损失（0-1 损失的代理） | 交叉熵损失（对数损失） |
| 可导性 | 损失函数在 $z = yf(\mathbf{x}) = 0$ 处不可导 | 处处可导 |
| 正则化 | 支持 L1/L2/ElasticNet | 支持 L1/L2/ElasticNet |
| 概率输出 | 无 | 有（Sigmoid） |

### 5. 在线学习

**代码对应**：`perc_online.partial_fit(X_shuffled, y_shuffled, classes=classes)` 实现增量学习。

感知机天然支持在线学习：每看到一个新样本就立即更新权重。这适合流式数据场景。

更新规则与批量训练相同，只是逐样本进行。多轮迭代（epoch）通常能提升效果。

### 6. 正则化

**代码对应**：`penalty` 参数支持 L1、L2、ElasticNet。

正则化后的更新：

$$\mathbf{w} \leftarrow \mathbf{w} + \eta(y_i\mathbf{x}_i - \alpha \cdot \nabla R(\mathbf{w}))$$

其中 $R(\mathbf{w})$ 为正则化项。L1 正则化可以产生稀疏权重（特征选择）。

## 代码结构

| 段落 | 内容 |
|------|------|
| 数据 | 二分类合成数据 + Iris 多分类 |
| 基础感知机 | 训练、权重、迭代次数 |
| 学习率对比 | eta0 从 0.001 到 10 |
| 正则化对比 | L1、L2、ElasticNet |
| 在线学习 | partial_fit 增量更新 |
| 局限性 | 非线性可分数据上的表现 |

### 1. 线性可分数据

运行 1. 线性可分数据 的代码，观察算法在该环节的行为。

In [ ]:
X, y = make_classification(
    n_samples=300, n_features=10, n_informative=5,
    n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 感知机对特征缩放敏感
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 2. 基础感知机

运行 2. 基础感知机 的代码，观察算法在该环节的行为。

In [ ]:
# Perceptron 本质上是随机梯度下降的线性模型
# eta0: 学习率, max_iter: 最大迭代次数
perc = Perceptron(eta0=1.0, max_iter=1000, random_state=42, early_stopping=True)
perc.fit(X_train, y_train)

print("=== 感知机分类 ===")
print(f"训练集准确率: {perc.score(X_train, y_train):.4f}")
print(f"测试集准确率: {perc.score(X_test, y_test):.4f}")
print(f"迭代次数: {perc.n_iter_}")
print(f"权重形状: {perc.coef_.shape}")
# --- 输出结果 ---
print(f"截距: {perc.intercept_}")

### 3. 学习率影响

运行 3. 学习率影响 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 学习率 (eta0) 对比 ===")
for eta in [0.001, 0.01, 0.1, 1.0, 10.0]:
    perc_e = Perceptron(eta0=eta, max_iter=1000, random_state=42)
    perc_e.fit(X_train, y_train)
    print(f"  eta0={eta:>6}: 训练={perc_e.score(X_train, y_train):.4f}, "
# --- 继续 ---
          f"测试={perc_e.score(X_test, y_test):.4f}, 迭代={perc_e.n_iter_}")

### 4. 正则化（penalty）

运行 4. 正则化（penalty） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 正则化对比 ===")
for penalty, alpha in [("l2", 0.0001), ("l2", 0.01), ("l1", 0.0001), ("l1", 0.01), ("elasticnet", 0.0001)]:
    perc_p = Perceptron(penalty=penalty, alpha=alpha, random_state=42)
    perc_p.fit(X_train, y_train)
    test_acc = perc_p.score(X_test, y_test)
# --- 数值计算 ---
    n_nonzero = np.count_nonzero(perc_p.coef_)
    print(f"  {penalty:>12} alpha={alpha}: 测试={test_acc:.4f}, 非零权重={n_nonzero}/{perc_p.coef_.shape[1]}")

### 5. 在线学习（partial_fit）

运行 5. 在线学习（partial_fit） 的代码，观察算法在该环节的行为。

In [ ]:
# 感知机支持增量学习，适合流式数据
print("\n=== 在线学习（partial_fit）===")
perc_online = Perceptron(eta0=0.1, max_iter=1, random_state=42)
classes = np.unique(y_train)

for epoch in range(10):
    perm = np.random.permutation(len(X_train))
    X_shuffled = X_train[perm]
    y_shuffled = y_train[perm]
    perc_online.partial_fit(X_shuffled, y_shuffled, classes=classes)
# --- 继续 ---
    acc = perc_online.score(X_test, y_test)
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1:>2}: 测试准确率={acc:.4f}")

### 6. 多分类（Iris）

在分类任务上训练模型并评估性能。

In [ ]:
print("\n=== 感知机在 Iris 上（多分类用 OvR）===")
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_i, y_i, test_size=0.2, random_state=42, stratify=y_i)
scaler_i = StandardScaler()
# --- 继续 ---
X_tr = scaler_i.fit_transform(X_tr)
X_te = scaler_i.transform(X_te)

perc_iris = Perceptron(max_iter=1000, random_state=42, early_stopping=True)
perc_iris.fit(X_tr, y_tr)
print(f"训练准确率: {perc_iris.score(X_tr, y_tr):.4f}")
print(f"测试准确率: {perc_iris.score(X_te, y_te):.4f}")
print(f"\n分类报告:\n{classification_report(y_te, perc_iris.predict(X_te), target_names=iris.target_names)}")

### 7. 感知机的局限性

运行 7. 感知机的局限性 的代码，观察算法在该环节的行为。

In [ ]:
X_xor, y_xor = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=2, random_state=42
)
perc_xor = Perceptron(max_iter=1000, random_state=42)
# --- 训练模型 ---
perc_xor.fit(X_xor, y_xor)
print(f"\n=== 感知机的局限（非线性可分数据）===")
print(f"测试准确率: {perc_xor.score(X_xor, y_xor):.4f}")
print("感知机只能处理线性可分问题，遇到 XOR 等非线性数据无法收敛")

print("\n=== 感知机要点 ===")
print("- 最早的神经网络模型（单层，无隐藏层）")
print("- 只能处理线性可分问题，非线性数据需要多层感知机（MLP）")
print("- 支持 partial_fit：可处理流式/在线学习场景")
print("- 收敛性：若数据线性可分，保证在有限步内收敛")
# --- 输出结果 ---
print("- 对学习率敏感：太小收敛慢，太大可能震荡不收敛")
print("- 与逻辑回归的区别：感知机直接输出类别，逻辑回归输出概率")

## 关键代码解释

### 在线学习（partial_fit）

```python
perc_online = Perceptron(eta0=0.1, max_iter=1, random_state=42)
for epoch in range(10):
    perc_online.partial_fit(X_shuffled, y_shuffled, classes=classes)
```

partial_fit 允许逐批更新模型，不需要一次性加载全部数据。这是**流式学习**（Online Learning）的基础——新数据来了就更新模型，不用从头训练。

### 正则化与稀疏化

```python
for penalty, alpha in [("l2", 0.0001), ("l1", 0.0001), ("elasticnet", 0.0001)]:
    perc_p = Perceptron(penalty=penalty, alpha=alpha, random_state=42)
```

L1 正则化可以把部分权重压缩到 0，实现特征选择。但感知机的正则化效果不如逻辑回归稳定，因为感知机的损失函数是误分类点到超平面的距离，不像逻辑回归的交叉熵损失那样平滑。

### 感知机 vs 逻辑回归

感知机输出类别（+1/-1），逻辑回归输出概率。感知机的决策边界取决于"最后被修正的样本"，逻辑回归的决策边界取决于"所有样本的概率分布"。在实践中，逻辑回归几乎总是比感知机更好用。

## 使用示例

```python
from sklearn.linear_model import Perceptron

perc = Perceptron(eta0=0.1, max_iter=1000, random_state=42)
perc.fit(X_train, y_train)
print(perc.predict(X_test))
```

## 注意事项

1. **只能线性可分**：感知机无法处理 XOR 等非线性问题，需要多层网络
2. **对学习率极其敏感**：太小收敛慢，太大不收敛（在最优解附近震荡）
3. **不输出概率**：没有 predict_proba 方法
4. **不保证唯一解**：不同的初始化可能得到不同的超平面
5. **收敛定理**：如果数据线性可分，感知机保证在有限步内收敛

## 总结与延伸

以上代码展示了 **感知机** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **核感知机**：加入核技巧，处理非线性问题（本质上接近 SVM）
- **Voted Perceptron**：保留所有中间权重向量并加权投票，比标准感知机更稳定
- **多层感知机（MLP）**：加隐藏层 → 非线性激活 → 反向传播，就是现代神经网络
- **感知机与 SGD 的关系**：sklearn 的 Perceptron 本质上就是 SGDClassifier(loss="perceptron")
